### 1\. Структура абелевих груп

Розглянемо абелеві групи порядку 400:
Z_400, Z_16 x Z_25, Z_20 x Z_20, Z_4 x Z_100, Z_2 x Z_10 x Z_20, Z_505^*.

Для кожної з цих груп виконайте наступні кроки:

1. Знайдіть елементарні дільники групи.
2. Розкладіть групу у добуток циклічних p-груп.
3. Визначте кількість елементів кожного можливого порядку.
4. Порівняйте групи та визначте, які з них є ізоморфними.



In [1]:
from math import gcd
from sympy import factorint

# --- Допоміжні функції ---

def cyclic_group_order_counts(n):
    """Кількість елементів кожного порядку в Z_n."""
    cnt = {}
    for k in range(n):
        # порядок елемента k в Z_n = n / gcd(n,k)
        order = n // gcd(n, k)
        cnt[order] = cnt.get(order, 0) + 1
    return cnt

def lcm(a, b):
    return a * b // gcd(a, b)

def direct_product_order_counts(orders):
    """Кількість елементів кожного порядку в прямому добутку циклічних груп."""
    comp_counts = [cyclic_group_order_counts(o) for o in orders]
    current = {1: 1}  # порядок -> кількість
    
    for cc in comp_counts:
        new = {}
        for p1, c1 in current.items():
            for p2, c2 in cc.items():
                new_order = lcm(p1, p2)
                new[new_order] = new.get(new_order, 0) + c1 * c2
        current = new
    return current

def elementary_divisors_from_cyclic_orders(orders):
    """Елементарні дільники для абелевої групи, заданої як добуток циклічних порядків."""
    result = []
    for o in orders:
        for p, e in factorint(o).items():
            result.append(p**e)
    return sorted(result)

def primary_decomposition(orders):
    """Розклад у добуток циклічних p-груп."""
    by_p = {}
    for o in orders:
        for p, e in factorint(o).items():
            if p not in by_p:
                by_p[p] = []
            by_p[p].append(p**e)
    for p in by_p:
        by_p[p] = sorted(by_p[p])
    return by_p

# --- Групи, які треба розглянути ---
groups = {
    "Z_400":                    [400],
    "Z_16 x Z_25":              [16, 25],
    "Z_20 x Z_20":              [20, 20],
    "Z_4 x Z_100":              [4, 100],
    "Z_2 x Z_10 x Z_20":        [2, 10, 20],
    "Z_505^*":                  [4, 100],  # Z_505^* ~ Z_4 x Z_100
}

# --- Обробка ---
results = {}
for name, orders in groups.items():
    print("=" * 60)
    total_order = 1
    for o in orders:
        total_order *= o
    print(f"Група: {name}  (порядок = {total_order})")
    
    ed = elementary_divisors_from_cyclic_orders(orders)
    print(f"  Елементарні дільники: {ed}")
    
    pd = primary_decomposition(orders)
    print(f"  Розклад на p-групи: ", end="")
    parts = []
    for p in sorted(pd):
        parts.append(" x ".join(f"Z_{q}" for q in pd[p]))
    print(" x ".join(parts))
    
    cnt = direct_product_order_counts(orders)
    print("  Кількість елементів кожного порядку:")
    for d in sorted(cnt):
        print(f"    порядок {d}: {cnt[d]}")
        
    results[name] = (tuple(ed), tuple(sorted(cnt.items())))

# --- Порівняння та ізоморфізм ---
print("\n" + "=" * 60)
print("ПОРІВНЯННЯ ГРУП (ізоморфізм за елементарними дільниками):")
names = list(results.keys())
for i in range(len(names)):
    for j in range(i+1, len(names)):
        n1, n2 = names[i], names[j]
        same_ed = results[n1][0] == results[n2][0]
        if same_ed:
            print(f"  {n1}  ~  {n2}   (ізоморфні)")
        else:
            print(f"  {n1}  !~ {n2}   (не ізоморфні)")

Група: Z_400  (порядок = 400)
  Елементарні дільники: [16, 25]
  Розклад на p-групи: Z_16 x Z_25
  Кількість елементів кожного порядку:
    порядок 1: 1
    порядок 2: 1
    порядок 4: 2
    порядок 5: 4
    порядок 8: 4
    порядок 10: 4
    порядок 16: 8
    порядок 20: 8
    порядок 25: 20
    порядок 40: 16
    порядок 50: 20
    порядок 80: 32
    порядок 100: 40
    порядок 200: 80
    порядок 400: 160
Група: Z_16 x Z_25  (порядок = 400)
  Елементарні дільники: [16, 25]
  Розклад на p-групи: Z_16 x Z_25
  Кількість елементів кожного порядку:
    порядок 1: 1
    порядок 2: 1
    порядок 4: 2
    порядок 5: 4
    порядок 8: 4
    порядок 10: 4
    порядок 16: 8
    порядок 20: 8
    порядок 25: 20
    порядок 40: 16
    порядок 50: 20
    порядок 80: 32
    порядок 100: 40
    порядок 200: 80
    порядок 400: 160
Група: Z_20 x Z_20  (порядок = 400)
  Елементарні дільники: [4, 4, 5, 5]
  Розклад на p-групи: Z_4 x Z_4 x Z_5 x Z_5
  Кількість елементів кожного порядку:
    порядок 1:

### 2\. Одна абелева група порядку n

1. Реалізуйте функцію, яка для заданого натурального числа n обчислює кількість попарно неізоморфних абелевих груп порядку n.
2. Знайдіть перші сто таких натуральних чисел n, для яких існує рівно одна абелева група порядку n. Для кожного знайденого n проаналізуйте його розклад на прості множники.
3. Сформулюйте гіпотезу: для яких n ∈ N існує рівно одна (з точністю до ізоморфізму) абелева група порядку n.



In [ ]:
from sympy import factorint

def partition_count(n):
    """Кількість розбиттів числа n (p(n))."""
    # динамічне програмування
    dp = [0] * (n + 1)
    dp[0] = 1
    for k in range(1, n + 1):
        for s in range(k, n + 1):
            dp[s] += dp[s - k]
    return dp[n]

def num_abelian_groups(n):
    """
    Кількість попарно неізоморфних абелевих груп порядку n.
    За теоремою: якщо n = p1^a1 * ... * pk^ak, то
    кількість = p(a1) * p(a2) * ... * p(ak),
    де p(a) — кількість розбиттів числа a.
    """
    if n == 1:
        return 1
    result = 1
    for p, a in factorint(n).items():
        result *= partition_count(a)
    return result

# --- 1. Перевірка функції ---
print("Перевірка num_abelian_groups:")
for n in [1, 2, 4, 8, 12, 16, 30, 36, 72, 100, 400]:
    print(f"  n={n}: {num_abelian_groups(n)} абелевих груп")

# --- 2. Перші 100 чисел n, для яких рівно одна абелева група ---
print("\nПерші 100 n з рівно однією абелевою групою:")
found = []
n = 1
while len(found) < 100:
    if num_abelian_groups(n) == 1:
        found.append(n)
    n += 1

for i, n in enumerate(found, 1):
    fac = factorint(n)
    fac_str = " * ".join(f"{p}^{a}" if a > 1 else f"{p}" for p, a in sorted(fac.items()))
    print(f"  {i:3d}. n = {n:5d}  =  {fac_str}")

# --- 3. Аналіз: усі показники степенів = 1 ---
print("\nПеревірка гіпотези: n вільне від квадратів (усі показники = 1)?")
all_squarefree = all(
    all(a == 1 for a in factorint(n).values()) for n in found
)
print(f"  Усі знайдені n вільні від квадратів: {all_squarefree}")

# Перевіримо обернене: чи кожне вільне від квадратів n дає рівно одну групу?
print("\nПерші 20 вільних від квадратів n та кількість груп:")
sqfree = [m for m in range(1, 200) if all(a == 1 for a in factorint(m).values())]
for m in sqfree[:20]:
    print(f"  n={m}: {num_abelian_groups(m)} груп")

### 3\. Дільники нуля та одиниці в скінченних кільцях

Розгляньте скінченні кільця Z_30, Z_12 x Z_15, M_3(Z_3) та для кожного з них виконайте наступні кроки:

1. Знайдіть дільники нуля та їх кількість.
2. Знайдіть дільники одиниці та їх кількість.
3. Порівняйте отримані результати та зробіть висновок про взаємозв'язок між дільниками нуля та дільниками одиниці в скінченних кільцях.



In [ ]:
from sage.all import *

def analyze_ring_elements(elements, zero, one, name, label_fmt=str):
    """
    Універсальний аналіз скінченного кільця, заданого списком елементів.
    elements — список елементів
    zero, one — нуль та одиниця
    """
    print("=" * 60)
    print(f"Кільце: {name}")
    n = len(elements)
    print(f"  Кількість елементів: {n}")

    # оборотні елементи
    units = []
    for a in elements:
        try:
            a.inverse_of_unit()
            units.append(a)
        except Exception:
            pass
    unit_set = set(units)

    # дільники нуля
    zero_divs = []
    for a in elements:
        if a == zero:
            continue
        for b in elements:
            if b != zero and a * b == zero:
                zero_divs.append(a)
                break
    zd_set = set(zero_divs)

    print(f"  Дільники одиниці (оборотні): {len(units)}")
    if len(units) <= 20:
        print(f"    {[label_fmt(u) for u in units]}")
    print(f"  Дільники нуля (ненульові): {len(zero_divs)}")
    if len(zero_divs) <= 20:
        print(f"    {[label_fmt(u) for u in zero_divs]}")

    # перевірка: кожен ненульовий елемент — або оборотний, або дільник нуля
    uncovered = []
    for a in elements:
        if a == zero:
            continue
        if a not in unit_set and a not in zd_set:
            uncovered.append(a)
    print(f"  Ненульових елементів, що не є ні оборотними, ні дільниками нуля: {len(uncovered)}")

    return {
        "name": name, "order": n,
        "num_units": len(units), "num_zero_divisors": len(zero_divs),
        "units": units, "zero_divs": zero_divs,
    }

# ---------- 1) Z_30 ----------
R1 = Integers(30)
res1 = analyze_ring_elements(list(R1), R1.zero(), R1.one(), "Z_30")

# ---------- 2) Z_12 x Z_15 (декартів добуток кілець) ----------
# Правильний спосіб у Sage — CartesianProduct зі структурою кільця
A = Integers(12)
B = Integers(15)
R2 = cartesian_product([A, B])          # або A.cartesian_product(B)
print("\nТип R2:", type(R2))

# елементи R2 — це кортежі (a, b)
elems2 = list(R2)
zero2 = R2.zero()
one2  = R2.one()
res2 = analyze_ring_elements(elems2, zero2, one2, "Z_12 x Z_15",
                             label_fmt=lambda t: f"({t[0]},{t[1]})")

# ---------- 3) M_3(Z_3) ----------
print("\n" + "=" * 60)
print("Кільце: M_3(Z_3)")
MS = MatrixSpace(GF(3), 3, 3)
mats = list(MS)
zero3 = MS.zero()
one3  = MS.one()

units3 = [M for M in mats if M.is_invertible()]
# Замість set(units3) використовуємо звичайний список
# (матриці unhashable, тому set не працює)

zero_divs3 = []
for M in mats:
    if M == zero3:
        continue
    for N in mats:
        if N != zero3 and M * N == zero3:
            zero_divs3.append(M)
            break

print(f"  Кількість елементів: {len(mats)}")
print(f"  Дільники одиниці (оборотні): {len(units3)}")
print(f"  Дільники нуля (ненульові): {len(zero_divs3)}")
print(f"  Сума оборотних + дільників нуля = {len(units3) + len(zero_divs3)} "
      f"(ненульових елементів: {len(mats) - 1})")

# ---------- Підсумок ----------
print("\n" + "=" * 60)
print("ПІДСУМОК:")
print(f"  Z_30:        порядок {res1['order']:3d}, оборотних {res1['num_units']:3d}, дільників нуля {res1['num_zero_divisors']:3d}")
print(f"  Z_12 x Z_15: порядок {res2['order']:3d}, оборотних {res2['num_units']:3d}, дільників нуля {res2['num_zero_divisors']:3d}")
print(f"  M_3(Z_3):    порядок {len(mats):4d}, оборотних {len(units3):4d}, дільників нуля {len(zero_divs3):4d}")

### 4\. Факторкільця кільця многочленів

Нехай f(x) = x^4 + x + 2. Виконайте наступні кроки для кожного з кілець R = Q, Z_4, Z_5:

1. Побудуйте ідеал I = (f(x)) у кільці R[x].
2. Побудуйте факторкільце R/I та знайдіть його порядок.
3. Визначте, чи є факторкільце R/I полем.
4. Знайдіть дільники нуля у факторкільці R/I.
5. Розкладіть f(x) на лінійні множники над R та порівняйте з дільниками нуля у факторкільці R/I.
6. Перевірте, чи елемент x+I ∈ R/I є оборотним.



In [ ]:
# ============================================================
# Задача 4. Факторкільця R[x]/(f)   (виправлена версія)
# ============================================================
from sage.all import *

def analyze_quotient(base_ring, base_name):
    print("=" * 60)
    print(f"R = {base_name}")
    R = base_ring
    P = PolynomialRing(R, 'x')
    x = P.gen()
    f = x**4 + x + 2
    print(f"  f(x) = {f}")

    # 1. Ідеал I = (f)
    I = P.ideal(f)
    print(f"  1. Ідеал I = (f) побудовано.")

    # 2. Факторкільце
    Q = P.quotient(I, 'a')
    a = Q.gen()
    print(f"  2. Факторкільце R/I = {Q}")

    # Визначаємо скінченність БЕЗ list(Q)
    finite = R.is_finite()
    if finite:
        order = R.order() ** f.degree()
        print(f"     Порядок R/I = {order}")
    else:
        order = None
        print(f"     Порядок R/I: нескінченний (R = {base_name})")

    # 3. Чи є поле?
    is_field = Q.is_field()
    print(f"  3. Чи є R/I полем? {is_field}")

    # 4. Дільники нуля
    zero_divs = []
    if finite:
        elems = list(Q)          # тепер безпечно, бо R скінченне
        zero = Q.zero()
        for el in elems:
            if el == zero:
                continue
            for b in elems:
                if b != zero and el * b == zero:
                    zero_divs.append(el)
                    break
        print(f"  4. Дільники нуля (ненульові): {len(zero_divs)}")
        if len(zero_divs) <= 30:
            print(f"     {zero_divs}")
        else:
            print(f"     {zero_divs[:30]} ...")
    else:
        # Для Q[x]/(f): дільники нуля є тоді й лише тоді, коли f звідний над Q
        fac = f.factor()
        print(f"  4. R/I нескінченне. Факторизація f над Q: {fac}")
        if len(fac) > 1 or fac[0][1] > 1:
            print(f"     f звідний над Q => R/I має дільники нуля "
                  f"(наприклад, класи множників f)")
        else:
            print(f"     f незвідний над Q => R/I — поле, дільників нуля немає")

    # 5. Розклад f на множники над R
    print(f"  5. Факторизація f над {base_name}:")
    try:
        fac = f.factor()
        print(f"     {fac}")
        all_linear = all(m.degree() == 1 for m, _ in fac)
        print(f"     Усі множники лінійні: {all_linear}")
    except Exception as e:
        fac = None
        print(f"     Факторизація недоступна над {base_name}: {e}")
        # для Z_4 перевіримо корені перебором
        if finite:
            roots = [r for r in R if f(r) == 0]
            print(f"     Корені f у {base_name}: {roots}")

    # 6. Чи оборотний x+I?
    try:
        inv = a.inverse_of_unit()
        print(f"  6. x+I оборотний, обернений: {inv}")
        x_unit = True
    except Exception:
        print(f"  6. x+I НЕ є оборотним")
        x_unit = False

    return {
        "name": base_name,
        "field": is_field,
        "order": order,
        "factorization": fac,
        "x_plus_I_unit": x_unit,
        "num_zero_divisors": len(zero_divs) if finite else None,
    }

# --- Q ---
res_Q  = analyze_quotient(QQ, "Q")

# --- Z_4 ---
res_Z4 = analyze_quotient(Integers(4), "Z_4")

# --- Z_5 ---
res_Z5 = analyze_quotient(Integers(5), "Z_5")

# --- Підсумок ---
print("\n" + "=" * 60)
print("ПІДСУМОК:")
for r in [res_Q, res_Z4, res_Z5]:
    print(f"  R = {r['name']:4s}: поле? {r['field']}, порядок: {r['order']}, "
          f"дільників нуля: {r['num_zero_divisors']}, "
          f"x+I оборотний? {r['x_plus_I_unit']}")
    print(f"        факторизація f: {r['factorization']}")